# 06 — Method Mode: Running Electrochemical Experiments

**Method mode** is the standard workflow for running complete electrochemical experiments (LSV, CV, EIS, chronoamperometry, etc.). You load a method file created in IviumSoft, optionally modify its parameters from Python, start it, monitor progress, and retrieve the measured data.

### Workflow overview

```
load_method()          ← load a .imf method file into IviumSoft
    ↓
set_method_parameter() ← (optional) modify parameters programmatically
    ↓
start_method()         ← start the measurement
    ↓
get_available_data_points_number()  ← poll for progress in a loop
    ↓
get_data_point(index)  ← read data during or after the run
    ↓
save_data()            ← save results to .idf file
```

### Prerequisites
- IviumSoft installed, running, device connected
- A method file (`.imf`) created in IviumSoft — see `C:/IviumStat/AppMethods/` for built-in examples

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyvium import Pyvium
from pyvium.errors import DeviceBusyError

Pyvium.open_driver()
Pyvium.connect_device()
print("Ready")

## 1. Loading a Method File

`load_method()` loads a `.imf` method file into IviumSoft without starting it. This lets you inspect or modify parameters before running.

> Update `METHOD_PATH` to point to a method file on your system.

In [ ]:
METHOD_PATH = r"C:/IviumStat/AppMethods/LSV.imf"  # update this path

try:
    Pyvium.load_method(METHOD_PATH)
    print(f"Loaded: {METHOD_PATH}")
except FileNotFoundError:
    print(f"Method file not found: {METHOD_PATH}")
    print("Update METHOD_PATH to a valid .imf file on your system")

## 2. Modifying Method Parameters

`set_method_parameter(name, value)` changes individual parameters of the loaded method. Both `name` and `value` are strings.

> **Important rules (from the IviumSoft DLL reference):**
> - Parameter names are **case-sensitive** and must exactly match the field label visible in the IviumSoft method tab.
> - Only **top-level visible parameters** can be changed — parameters hidden behind a checkbox or in a pop-up are not accessible.
> - To change the method type, set `'Method'` first, then `'Technique'`. If you set them in the wrong order the change is silently ignored.
> - Invalid parameter names or values are silently ignored (no exception is raised).

### Changing the method type

```python
# Always set Method before Technique
Pyvium.set_method_parameter('Method',    'CyclicVoltammetry')
Pyvium.set_method_parameter('Technique', 'Standard')
```

### Common scan parameters (names must match the IviumSoft UI exactly)

| Parameter name | Typical method | Example value |
|----------------|---------------|---------------|
| `'Method'` | any | `'CyclicVoltammetry'`, `'LinearSweep'`, `'Potentiostatic'` |
| `'Technique'` | any | `'Standard'`, `'SampledCurrent'` |
| `'E begin'` | LSV, CV | `'0.0'` |
| `'E end'` | LSV | `'1.0'` |
| `'E step'` | LSV | `'0.005'` |
| `'scanrate'` | CV | `'0.05'` |
| `'duration'` | Chronoamperometry | `'10'` |
| `'freq high'` | EIS | `'100000'` |
| `'freq low'` | EIS | `'0.1'` |

> Exact names depend on the loaded method. Open the method in IviumSoft and read the field labels directly to confirm the spelling.

In [ ]:
# Example: modify scan start and end potential for an LSV
Pyvium.set_method_parameter("E begin", "0.0")
Pyvium.set_method_parameter("E end",   "0.8")
Pyvium.set_method_parameter("E step",  "0.005")
print("Parameters updated: LSV from 0.0 V to 0.8 V, step 5 mV")

## 3. Saving the Modified Method

After modifying parameters you can save the method back to disk. This is useful for building a library of parameterized method templates from Python.

In [ ]:
SAVE_METHOD_PATH = r"C:/IviumStat/AppMethods/LSV_modified.imf"

Pyvium.save_method(SAVE_METHOD_PATH)
print(f"Method saved to: {SAVE_METHOD_PATH}")

## 4. Starting a Method

`start_method()` has two modes:

- `start_method()` — starts the method currently loaded in IviumSoft
- `start_method(path)` — loads and starts a method from a file path in one call

The method runs asynchronously — the call returns immediately and the device starts measuring.

In [ ]:
# Option A: start the currently loaded method
Pyvium.start_method()
print("Method started")

# Option B: load and start in one call
# Pyvium.start_method(r"C:/IviumStat/AppMethods/LSV.imf")

## 5. Monitoring Progress

`get_available_data_points_number()` returns how many data points have been recorded so far. Poll it in a loop to track progress and read data in real time.

In [ ]:
TOTAL_EXPECTED_POINTS = 160  # set to match your method's expected point count
POLL_INTERVAL_S       = 0.1

while True:
    n_points = Pyvium.get_available_data_points_number()
    progress = min(100.0, 100.0 * n_points / TOTAL_EXPECTED_POINTS)
    print(f"\rProgress: {n_points} points ({progress:.0f}%)", end="", flush=True)

    if n_points >= TOTAL_EXPECTED_POINTS:
        break
    time.sleep(POLL_INTERVAL_S)

print("\nMeasurement complete")

## 6. Reading Data Points

`get_data_point(index)` returns three values whose meaning depends on the method type:

| Method type | Value 1 | Value 2 | Value 3 |
|-------------|---------|---------|--------|
| LSV / CV | Potential (V) | Current (A) | 0 |
| Chronoamperometry | Time (s) | Current (A) | Potential (V) |
| EIS | Re(Z) (Ω) | Im(Z) (Ω) | Frequency (Hz) |
| Chronopotentiometry | Time (s) | Potential (V) | Current (A) |

Index is 1-based.

In [ ]:
# Read all available data points
total_points = Pyvium.get_available_data_points_number()

data = []
for i in range(1, total_points + 1):
    v1, v2, v3 = Pyvium.get_data_point(i)
    data.append([v1, v2, v3])

# Build a DataFrame — column names depend on your method type
df = pd.DataFrame(data, columns=["Potential (V)", "Current (A)", "_unused"])
print(f"Read {len(df)} data points")
print(df.head())

## 7. Plotting Results — LSV Example

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["Potential (V)"], df["Current (A)"] * 1e6, 'b-', lw=1.5)
ax.set_xlabel("Potential (V)")
ax.set_ylabel("Current (µA)")
ax.set_title("Linear Sweep Voltammetry")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. EIS Example — Nyquist Plot

In [ ]:
# Load and run an EIS method, then retrieve data
EIS_METHOD = r"C:/IviumStat/AppMethods/EIS.imf"

# Pyvium.start_method(EIS_METHOD)
# ... wait for completion ...

total = Pyvium.get_available_data_points_number()
eis_data = []
for i in range(1, total + 1):
    Z_re, Z_im, freq = Pyvium.get_data_point(i)
    eis_data.append([freq, Z_re, Z_im])

eis_df = pd.DataFrame(eis_data, columns=["Frequency (Hz)", "Z_re (Ω)", "Z_im (Ω)"])

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(eis_df["Z_re (Ω)"], -eis_df["Z_im (Ω)"], 'ro-', markersize=4)
ax.set_xlabel("Z' (Ω)")
ax.set_ylabel("-Z'' (Ω)")
ax.set_title("Nyquist Plot")
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Reading Multi-Scan Data (CV)

For cyclic voltammetry (CV) with multiple scans, `get_data_point_from_scan(point_index, scan_index)` lets you retrieve data from any scan — not just the last one.

Both `point_index` and `scan_index` are 1-based.

In [ ]:
CV_METHOD       = r"C:/IviumStat/AppMethods/CV.imf"
N_SCANS         = 3
POINTS_PER_SCAN = 200  # update to match your method

# Pyvium.start_method(CV_METHOD)
# ... wait for all scans to finish ...

fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.viridis([i / N_SCANS for i in range(N_SCANS)])

for scan in range(1, N_SCANS + 1):
    potentials, currents = [], []
    for pt in range(1, POINTS_PER_SCAN + 1):
        E, I, _ = Pyvium.get_data_point_from_scan(pt, scan)
        potentials.append(E)
        currents.append(I * 1e6)  # µA
    ax.plot(potentials, currents, color=colors[scan - 1], label=f"Scan {scan}")

ax.set_xlabel("Potential (V)")
ax.set_ylabel("Current (µA)")
ax.set_title(f"Cyclic Voltammetry — {N_SCANS} scans")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Aborting a Running Method

In [ ]:
# Call this while a method is running to stop it immediately
# Pyvium.abort_method()
# print("Method aborted")

# Example: abort if measurement takes too long
TIMEOUT_S = 60

# Pyvium.start_method()
t_start = time.time()
while Pyvium.get_available_data_points_number() < TOTAL_EXPECTED_POINTS:
    if time.time() - t_start > TIMEOUT_S:
        Pyvium.abort_method()
        print("Timeout reached — method aborted")
        break
    time.sleep(0.5)
else:
    print("Method completed within timeout")

## 11. Saving Results

`save_data()` saves the last measurement to an IDF file. `save_dataset()` saves all measurements in the current measurement list.

> **Warning:** `save_data()` with an invalid path will **close IviumSoft**. Always use absolute paths and verify they exist first.

In [ ]:
import os

OUTPUT_DIR  = r"C:/IviumStat/Data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "my_measurement.idf")

if os.path.isdir(OUTPUT_DIR):
    Pyvium.save_data(OUTPUT_FILE)
    print(f"Data saved to: {OUTPUT_FILE}")
else:
    print(f"Output directory does not exist: {OUTPUT_DIR}")
    print("Create the directory first or update OUTPUT_DIR")

In [ ]:
# Save all measurements in the measurement list to a dataset file
DATASET_FILE = os.path.join(OUTPUT_DIR, "my_dataset.idf")

if os.path.isdir(OUTPUT_DIR):
    Pyvium.save_dataset(DATASET_FILE)
    print(f"Dataset saved to: {DATASET_FILE}")

## 12. Temperature Logging (Beta)

In multi-channel setups, `update_temperature()` broadcasts a temperature value to all channels. This is a metadata field — it does not control a thermostat.

In [ ]:
TEMPERATURE_CELSIUS = 25.0

Pyvium.update_temperature(TEMPERATURE_CELSIUS)
print(f"Temperature set to {TEMPERATURE_CELSIUS} °C")

## 13. SQL Database File

IviumSoft can store measurement data in an SQL database. `get_db_file_name()` returns the path to the last created database file.

In [ ]:
db_path = Pyvium.get_db_file_name()
print(f"SQL database: {db_path}")

## 14. Complete Experiment Workflow

Putting it all together: a robust template that handles errors, monitors progress, reads data, and saves results.

In [ ]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyvium import Pyvium
from pyvium.errors import DeviceBusyError, IviumSoftNotRunningError

METHOD_PATH      = r"C:/IviumStat/AppMethods/LSV.imf"
OUTPUT_PATH      = r"C:/IviumStat/Data/lsv_result.idf"
EXPECTED_POINTS  = 160
POLL_INTERVAL_S  = 0.2

def run_experiment(method_path, output_path, expected_points):
    print(f"Starting: {os.path.basename(method_path)}")
    Pyvium.start_method(method_path)

    # Monitor progress
    while True:
        n = Pyvium.get_available_data_points_number()
        print(f"\r  {n}/{expected_points} points", end="", flush=True)
        if n >= expected_points:
            break
        time.sleep(POLL_INTERVAL_S)
    print("  — complete")

    # Collect data
    rows = []
    for i in range(1, n + 1):
        rows.append(Pyvium.get_data_point(i))
    df = pd.DataFrame(rows, columns=["E (V)", "I (A)", "aux"])

    # Save
    if os.path.isdir(os.path.dirname(output_path)):
        Pyvium.save_data(output_path)
        print(f"  Saved → {output_path}")

    return df

try:
    Pyvium.open_driver()
    Pyvium.connect_device()

    df = run_experiment(METHOD_PATH, OUTPUT_PATH, EXPECTED_POINTS)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df["E (V)"], df["I (A)"] * 1e6, lw=1.5)
    ax.set_xlabel("Potential (V)")
    ax.set_ylabel("Current (µA)")
    ax.set_title("LSV Result")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

except IviumSoftNotRunningError:
    print("ERROR: IviumSoft is not running")
except DeviceBusyError:
    print("ERROR: Device is busy")
finally:
    try:
        Pyvium.close_driver()
    except Exception:
        pass

---

## Summary

| Task | Method |
|------|--------|
| Load method file | `load_method(path)` |
| Modify a parameter | `set_method_parameter(name, value)` |
| Save modified method | `save_method(path)` |
| Start method (loaded) | `start_method()` |
| Start method (from file) | `start_method(path)` |
| Abort running method | `abort_method()` |
| Check progress | `get_available_data_points_number()` |
| Read a data point | `get_data_point(index)` → (v1, v2, v3) |
| Read from a specific scan | `get_data_point_from_scan(pt, scan)` |
| Save last result | `save_data(path)` |
| Save full dataset | `save_dataset(path)` |
| Log temperature | `update_temperature(°C)` |
| Get SQL DB path | `get_db_file_name()` |

## Next

- **`07_data_processing.ipynb`** — Parse saved IDF files offline (no hardware needed)
- **`08_batch_and_synchronization.ipynb`** — Coordinate multiple devices simultaneously